In [1]:
%load_ext autoreload
%autoreload 2

# Forward Trespass

In [2]:
# Load the transitions from the forward_tres
import pickle
# Generated by test_tres.py

with open('../data/graph/transitions/LEMD_EGLL_2023_04_01_CLB.pkl', 'rb') as f:
    transitions_list = pickle.load(f)


In [3]:
# transitions_list = [(185, 0, 0.0, 546, 11, 14378.0),
#  (185, 0, 0.0, 472, 8, 11638.0),
#  (185, 0, 0.0, 491, 8, 10883.0),...]
#  (node_id, k_u_std_idx, alt_u_std, v_idx, k_v_std_idx, alt_v_std)

from_ids = set(t[0] for t in transitions_list)
to_ids = set(t[3] for t in transitions_list)
print(f"Number of unique from IDs: {len(from_ids)}")
print(f"Number of unique to IDs: {len(to_ids)}")


Number of unique from IDs: 47
Number of unique to IDs: 145


In [4]:
import networkx as nx
# Load the route graph
G = nx.read_gml("../data/graph/LEMD_EGLL_2023_04_01.gml")
node_to_idx = {node: i for i, node in enumerate(G.nodes())}
idx_to_node = {i: node for i, node in enumerate(G.nodes())}

In [5]:
def print_states_at_node_from_transitions(node_idx: int, transitions_list: list[tuple], idx_to_node: dict[int, str]):
    """
    Print a padded table of all the states with node_id == node_idx from transitions_list.
    transitions_list: list of (u_idx, k_u_std_idx, alt_u_std, v_idx, k_v_std_idx, alt_v_std)
    """
    headers = ["u_wp", "k_u_idx", "alt_u", "v_wp", "k_v_idx", "alt_v"]
    col_widths = [7, 8, 10, 7, 8, 10]
    # Print header
    header_row = "".join(h.ljust(w) for h, w in zip(headers, col_widths))
    print(header_row)
    print("-" * sum(col_widths))
    for t in transitions_list:
        if t[0] == node_idx:
            u_wp = idx_to_node.get(t[0], str(t[0]))
            v_wp = idx_to_node.get(t[3], str(t[3]))
            row = [
                str(u_wp).ljust(col_widths[0]),
                str(t[1]).ljust(col_widths[1]),
                f"{t[2]:.1f}".ljust(col_widths[2]),
                str(v_wp).ljust(col_widths[3]),
                str(t[4]).ljust(col_widths[4]),
                f"{t[5]:.1f}".ljust(col_widths[5]),
            ]
            print("".join(row))

print_states_at_node_from_transitions(node_to_idx['LEMD'], transitions_list, idx_to_node)

u_wp   k_u_idx alt_u     v_wp   k_v_idx alt_v     
--------------------------------------------------
LEMD   0       0.0       RBO    11      14378.0   
LEMD   0       0.0       RBO_16 8       11638.0   
LEMD   0       0.0       LECV   8       10883.0   
LEMD   0       0.0       LERM   11      14575.0   
LEMD   0       0.0       LETP   17      20473.0   
LEMD   0       0.0       PINAR  20      23041.0   
LEMD   0       0.0       EDIGO_3919      22708.0   
LEMD   0       0.0       OSTIX  23      26760.0   
LEMD   0       0.0       GASMO  27      29448.0   
LEMD   0       0.0       SIE_06 15      18398.0   
LEMD   0       0.0       SIE    16      19451.0   
LEMD   0       0.0       UNSOL_4520      23397.0   
LEMD   0       0.0       DISKO_3817      19963.0   
LEMD   0       0.0       SEGRE  23      25984.0   
LEMD   0       0.0       LETO   3       3750.0    
LEMD   0       0.0       BASIM_8420      23040.0   
LEMD   0       0.0       ZANKO  28      30149.0   
LEMD   0       0.0       AV

---
# Backward Trespass

In [6]:
# Load the transitions from the forward_tres
import pickle
# Generated by test_tres.py

with open('../data/graph/transitions/LEMD_EGLL_2023_04_01_CLSR.pkl', 'rb') as f:
    transitions_closure_list = pickle.load(f) # (u_idx, k_u_std_idx, rho_u_std_idx, alt_u_std, phase_u_std, v_idx, k_v_idx, rho_v_idx, alt_v_val, phi_v_idx)


In [7]:
with open('../data/graph/transitions/LEMD_EGLL_2023_04_01_CLB.pkl', 'rb') as f:
    climb_transitions_list = pickle.load(f)



In [8]:
import torch
import numpy as np

def print_states_at_node(node_idx: int, transitions_closure_list: list[tuple[int, int, int, float, int, int, int, int, float, int]],
                         idx_to_node: dict[int, str], mode='from'):
    # Define column headers and widths, replacing u_idx and v_idx with waypoint names
    headers = [
        "u_wp", "k_u_idx", "rho_u_idx", "alt_u", "phase_u",
        "v_wp", "k_v_idx", "rho_v_idx", "alt_v", "phase_v"
    ]
    col_widths = [7, 8, 10, 10, 8, 7, 8, 10, 10, 8]
    # Phase mapping
    phase_map = {0: "CLB", 1: "CRZ", 2: "DES"}
    # Print header
    header_row = "".join(h.ljust(w) for h, w in zip(headers, col_widths))
    print(header_row)
    print("-" * sum(col_widths))
    # Print each matching transition in padded columns, mapping indices to waypoint names
    for transition in transitions_closure_list:
        if mode == 'from':
            from_idx = 0
        elif mode == 'to':
            from_idx = 5
        else:
            raise ValueError(f"Invalid mode: {mode}")
        if transition[from_idx] == node_idx:
            phase_u_str = phase_map.get(transition[4], str(transition[4]))
            phase_v_str = phase_map.get(transition[9], str(transition[9]))
            u_wp = idx_to_node.get(transition[0], str(transition[0]))
            v_wp = idx_to_node.get(transition[5], str(transition[5]))
            row = (
                f"{u_wp:<7}{transition[1]:<8}{transition[2]:<10}"
                f"{transition[3]:<10.1f}{phase_u_str:<8}"
                f"{v_wp:<7}{transition[6]:<8}{transition[7]:<10}"
                f"{transition[8]:<10.1f}{phase_v_str:<8}"
            )
            print(row)

node_to_inspect = 'LEMD'
print(node_to_inspect)
print_states_at_node(node_to_idx[node_to_inspect], transitions_closure_list, idx_to_node)


LEMD
u_wp   k_u_idx rho_u_idx alt_u     phase_u v_wp   k_v_idx rho_v_idx alt_v     phase_v 
--------------------------------------------------------------------------------------
LEMD   39      0         35000.0   CRZ     NEDUS  43      0         35000.0   CRZ     
LEMD   39      36        0.0       CLB     NEDUS  43      0         35000.0   CRZ     
LEMD   39      36        0.0       CLB     BAKUP  43      0         35000.0   CRZ     
LEMD   39      0         35000.0   CRZ     BAKUP  45      0         35000.0   CRZ     
LEMD   40      0         35000.0   CRZ     BAKUP  46      0         35000.0   CRZ     
LEMD   40      0         35000.0   CRZ     BAKUP  47      0         35000.0   CRZ     
LEMD   39      0         35000.0   CRZ     NUBLO  43      0         35000.0   CRZ     
LEMD   39      36        0.0       CLB     NUBLO  43      0         35000.0   CRZ     
LEMD   40      0         35000.0   CRZ     NUBLO  44      0         35000.0   CRZ     
LEMD   39      0         35000.0   CRZ

In [9]:
print_states_at_node(node_to_idx['EGLL'], transitions_closure_list, idx_to_node, mode='to')


u_wp   k_u_idx rho_u_idx alt_u     phase_u v_wp   k_v_idx rho_v_idx alt_v     phase_v 
--------------------------------------------------------------------------------------
EGJA   56      0         32653.0   DES     EGLL   60      0         0.0       DES     
EGKA   57      0         20395.0   DES     EGLL   60      0         0.0       DES     
NEDUL  57      0         24399.0   DES     EGLL   60      0         0.0       DES     
GODIX  54      0         35000.0   CRZ     EGLL   60      0         0.0       DES     
PIGOP  54      0         35000.0   CRZ     EGLL   60      0         0.0       DES     
CPT    58      0         15849.0   DES     EGLL   60      0         0.0       DES     
EMKAD  57      0         21559.0   DES     EGLL   60      0         0.0       DES     
GODEM_0150      0         35000.0   CRZ     EGLL   60      0         0.0       DES     
MAXIT  58      0         8510.0    DES     EGLL   60      0         0.0       DES     
MIMFO  57      0         20329.0   DES    

---
# <font color='red'>Thinning and Wind Average Amortization</font>

In [10]:
from equinox.dp.trespass.thinning import thin_closures
from equinox.dp.trespass.tres_forward import save_transitions

# Option A: infer max_rho directly from the closure tuples.
thinned_closures = thin_closures(node_to_idx['LEMD'], node_to_idx['EGLL'], None, G, transitions_closure_list)
save_transitions(thinned_closures, '../data/graph/transitions', 'LEMD_EGLL_2023_04_01_REACHABLE')

Saved transitions to ../data/graph/transitions\LEMD_EGLL_2023_04_01_REACHABLE.pkl


In [ ]:
# Calculate the average wind for all transitions
from equinox.wind.wind_date import WindDate
from equinox.wind.wind_model import WindModel 
from equinox.helpers.datetimeh import datestr_to_seconds_since_midnight

wind_model = WindDate(date_str="2024-04-01", data_dir="data/era5")

delta_t_wall_clock_sec = 300.0  # 5 minutes
max_flight_duration_hours = 5.0
num_time_bins_wall_clock = int(max_flight_duration_hours * 3600 / delta_t_wall_clock_sec) + 1

# CRITICAL FIX: Use same time reference as tres_backward
estimated_landing_time_str = "2023-04-01 12:00:00"
estimated_landing_ssm = datestr_to_seconds_since_midnight(estimated_landing_time_str)
min_wall_clock_time_sec = float(estimated_landing_ssm - max_flight_duration_hours * 3600)

wind_model.get_average_wind_on_edges(transitions_closure_list, node_coords_deg, min_wall_clock_time_sec, delta_t_wall_clock_sec, num_integration_steps=3)

## Inspection

In [11]:
# REACHABLE States
print_states_at_node(node_to_idx['LEMD'], thinned_closures, idx_to_node, mode='from')

u_wp   k_u_idx rho_u_idx alt_u     phase_u v_wp   k_v_idx rho_v_idx alt_v     phase_v 
--------------------------------------------------------------------------------------
LEMD   39      36        0.0       CLB     NEDUS  43      0         35000.0   CRZ     
LEMD   39      36        0.0       CLB     BAKUP  43      0         35000.0   CRZ     
LEMD   39      36        0.0       CLB     NUBLO  43      0         35000.0   CRZ     
LEMD   39      36        0.0       CLB     SUSOS_4943      0         35000.0   CRZ     
LEMD   39      36        0.0       CLB     LFDA   43      0         35000.0   CRZ     
LEMD   39      36        0.0       CLB     LFBP   43      0         35000.0   CRZ     
LEMD   39      36        0.0       CLB     RATAS_8643      0         35000.0   CRZ     
LEMD   39      36        0.0       CLB     CALCE  42      0         35000.0   CRZ     
LEMD   39      36        0.0       CLB     CALCE  43      0         35000.0   CRZ     
LEMD   39      36        0.0       CLB   

In [12]:
# CLOSURE States
print_states_at_node(node_to_idx['ZANKO'], transitions_closure_list, idx_to_node, mode='to')

u_wp   k_u_idx rho_u_idx alt_u     phase_u v_wp   k_v_idx rho_v_idx alt_v     phase_v 
--------------------------------------------------------------------------------------
LEMD   39      0         35000.0   CRZ     ZANKO  41      0         35000.0   CRZ     
LEMD   39      36        0.0       CLB     ZANKO  41      8         30149.0   CLB     
LEMD   39      36        0.0       CLB     ZANKO  42      8         30149.0   CLB     


In [13]:
def print_states_at_node_from_transitions(node_idx: int, transitions_list: list[tuple[int, int, float, int, int, float]],
                                         idx_to_node: dict[int, str], mode='from'):
    # Define column headers and widths for the 6-element transition format
    headers = [
        "u_wp", "k_u_idx", "alt_u", "v_wp", "k_v_idx", "alt_v"
    ]
    col_widths = [7, 8, 10, 7, 8, 10]
    
    # Print header
    header_row = "".join(h.ljust(w) for h, w in zip(headers, col_widths))
    print(header_row)
    print("-" * sum(col_widths))
    
    # Print each matching transition in padded columns, mapping indices to waypoint names
    for transition in transitions_list:
        if mode == 'from':
            from_idx = 0
        elif mode == 'to':
            from_idx = 3
        else:
            raise ValueError(f"Invalid mode: {mode}")
            
        if transition[from_idx] == node_idx:
            u_wp = idx_to_node.get(transition[0], str(transition[0]))
            v_wp = idx_to_node.get(transition[3], str(transition[3]))
            row = (
                f"{u_wp:<7}{transition[1]:<8}{transition[2]:<10.1f}"
                f"{v_wp:<7}{transition[4]:<8}{transition[5]:<10.1f}"
            )
            print(row)
# Climb Transitions
print_states_at_node_from_transitions(node_to_idx['ZANKO'], climb_transitions_list, idx_to_node, mode='to')

u_wp   k_u_idx alt_u     v_wp   k_v_idx alt_v     
--------------------------------------------------
LEMD   0       0.0       ZANKO  28      30149.0   


---
# Forward Value Function Inspection

In [18]:
# Load the forward value function from sparse format
import numpy as np
import sys
import os
sys.path.append(os.path.abspath('../src'))
from equinox.dp.trespass.sparse_io_utils import load_sparse_coo_tensor_with_convention

# Load sparse tensor and convert to dense numpy array
V_fw_sparse, interpretation_note = load_sparse_coo_tensor_with_convention(
    '../data/graph/V_soft/LEMD_EGLL_2023_04_01_V_FWD_SPRSE.pt',
    target_device='cpu'
)

V_soft_fwd_sparse_coalesced = V_fw_sparse.coalesce()

V_soft_fwd_dense_filled = torch.full(
V_soft_fwd_sparse_coalesced.shape,
        -float('inf'),
        dtype=V_soft_fwd_sparse_coalesced.dtype,
        device=V_soft_fwd_sparse_coalesced.device
)

    # 3. Get indices and values from the coalesced sparse tensor.
indices = V_soft_fwd_sparse_coalesced.indices()
values = V_soft_fwd_sparse_coalesced.values()

# 4. Place the explicit values from the sparse tensor into the dense tensor.
#    This is done only if there are any explicit values.
if values.numel() > 0:
    V_soft_fwd_dense_filled[tuple(indices)] = values

# 5. Convert to numpy array on CPU.
V_soft_fwd_np = V_soft_fwd_dense_filled.cpu().numpy()


print(f"Loaded sparse forward value function. Interpretation: {interpretation_note}")
V_fw = V_soft_fwd_np
print(f"Forward value function shape: {V_fw.shape}")
print(f"Number of finite values: {np.isfinite(V_fw).sum()}")
V_fw.shape

Loaded sparse forward value function. Interpretation: Implicit zeros should be treated as float('inf'). Only explicitly stored values are actual costs.
Forward value function shape: (566, 61, 37, 3)
Number of finite values: 4088


(566, 61, 37, 3)

In [19]:
def print_value_function(node_name: str, node_to_idx: dict[str, int], V_fw: np.ndarray, mode='forward'):
    """
    Print all non-infinite values of the value function for a given node
    in a nicely formatted table.
    
    Args:
        node_name: Name of the waypoint node
        node_to_idx: Dictionary mapping node names to indices
        V_fw: Forward value function array with shape (waypoint_id, wall_clock_bin_id, climb_allowance_bin_id, phase)
    """
    if node_name not in node_to_idx:
        print(f"Node '{node_name}' not found in node_to_idx")
        return
    
    node_idx = node_to_idx[node_name]
    
    # Define column headers and widths
    headers = ["Node", "Wall_Clk", "Climb_Bin", "Phase", "Value"]
    col_widths = [8, 10, 11, 7, 12]
    
    # Print header
    header_row = "".join(h.ljust(w) for h, w in zip(headers, col_widths))
    print(f"{'Forward' if mode == 'forward' else 'Backward'} Value Function for node: {node_name} (idx: {node_idx})")
    print("=" * sum(col_widths))
    print(header_row)
    print("-" * sum(col_widths))
    
    # Find and print all non-infinite values
    finite_count = 0
    for k in range(V_fw.shape[1]):  # wall_clock_bin_id
        for rho in range(V_fw.shape[2]):  # climb_allowance_bin_id
            for phase in range(V_fw.shape[3]):  # phase
                value = V_fw[node_idx, k, rho, phase]
                if np.isfinite(value):
                    phase_name = {0: "CLB", 1: "CRZ", 2: "DES"}.get(phase, str(phase))
                    row = [
                        node_name.ljust(col_widths[0]),
                        str(k).ljust(col_widths[1]),
                        str(rho).ljust(col_widths[2]),
                        phase_name.ljust(col_widths[3]),
                        f"{value:.4f}".ljust(col_widths[4])
                    ]
                    print("".join(row))
                    finite_count += 1
    
    print("-" * sum(col_widths))
    print(f"Total finite values: {finite_count}")

# Example usage - print forward value function for LEMD
print_value_function('LEMD', node_to_idx, V_fw)


Forward Value Function for node: LEMD (idx: 185)
Node    Wall_Clk  Climb_Bin  Phase  Value       
------------------------------------------------
LEMD    39        36         CLB    1.3863      
LEMD    40        36         CLB    1.3863      
LEMD    41        36         CLB    1.3863      
LEMD    42        36         CLB    1.3863      
------------------------------------------------
Total finite values: 4


---
# Backward Value Function Inspection

In [20]:
# Load the backward value function from sparse format
import numpy as np 
# sparse_io_utils already imported in previous cell

# Load sparse tensor and convert to dense numpy array
V_bw_sparse, interpretation_note_bw = load_sparse_coo_tensor_with_convention(
    '../data/graph/V_soft/LEMD_EGLL_2023_04_01_V_BWD_SPRSE.pt',
    target_device='cpu'
)
print(f"Loaded sparse backward value function. Interpretation: {interpretation_note_bw}")

# New logic to fill unspecified sparse entries with -inf when converting to dense
# 1. Coalesce the sparse tensor (good practice, ensures unique indices).
V_bw_sparse_coalesced = V_bw_sparse.coalesce()

# 2. Create a dense tensor filled with -infinity.
#    Use the sparse tensor's dtype and device.
V_bw_dense_filled = torch.full(
    V_bw_sparse_coalesced.shape,
    -float('inf'),
    dtype=V_bw_sparse_coalesced.dtype,
    device=V_bw_sparse_coalesced.device
)

# 3. Get indices and values from the coalesced sparse tensor.
indices = V_bw_sparse_coalesced.indices()
values = V_bw_sparse_coalesced.values()

# 4. Place the explicit values from the sparse tensor into the dense tensor.
#    This is done only if there are any explicit values.
if values.numel() > 0:
    V_bw_dense_filled[tuple(indices)] = values

# 5. Convert to numpy array on CPU.
V_bw = V_bw_dense_filled.cpu().numpy()
print(f"Backward value function shape: {V_bw.shape}")
print(f"Number of finite values: {np.isfinite(V_bw).sum()}")
V_bw.shape

Loaded sparse backward value function. Interpretation: Implicit zeros should be treated as float('inf'). Only explicitly stored values are actual costs.
Backward value function shape: (566, 61, 37, 3)
Number of finite values: 4088


(566, 61, 37, 3)

In [21]:
print_value_function('LEMD', node_to_idx, V_bw, mode='backward')
print_value_function('EGLL', node_to_idx, V_bw, mode='backward')
print_value_function('LEMD', node_to_idx, V_fw)
print_value_function('EGLL', node_to_idx, V_fw)

Backward Value Function for node: LEMD (idx: 185)
Node    Wall_Clk  Climb_Bin  Phase  Value       
------------------------------------------------
LEMD    39        36         CLB    -35.8177    
LEMD    40        36         CLB    -36.2243    
LEMD    41        36         CLB    -35.4568    
LEMD    42        36         CLB    -31.4131    
------------------------------------------------
Total finite values: 4
Backward Value Function for node: EGLL (idx: 54)
Node    Wall_Clk  Climb_Bin  Phase  Value       
------------------------------------------------
EGLL    60        0          DES    -0.0000     
------------------------------------------------
Total finite values: 1
Forward Value Function for node: LEMD (idx: 185)
Node    Wall_Clk  Climb_Bin  Phase  Value       
------------------------------------------------
LEMD    39        36         CLB    1.3863      
LEMD    40        36         CLB    1.3863      
LEMD    41        36         CLB    1.3863      
LEMD    42        36  

Verification $V_f(g) = V_b(s)$:

In [22]:
def log_sum_exp(values):
    """
    Compute log(sum(exp(values))) in a numerically stable way.
    
    Args:
        values: List or array of log-space values
        
    Returns:
        The log-sum-exp of the input values
    """
    import numpy as np
    
    # Convert to numpy array if not already
    values = np.array(values)
    
    # Handle empty input
    if len(values) == 0:
        return -np.inf
    
    # Handle case where all values are -inf
    if np.all(np.isneginf(values)):
        return -np.inf
    
    # Find the maximum value for numerical stability
    max_val = np.max(values[np.isfinite(values)])
    
    # Subtract max_val from all values, compute exp, sum, then add max_val back
    # This prevents overflow/underflow issues
    shifted_values = values - max_val
    exp_sum = np.sum(np.exp(shifted_values[np.isfinite(shifted_values)]))
    
    return max_val + np.log(exp_sum)



In [23]:
log_sum_exp([35.8177 - 1.3863, 36.2243 - 1.3863, 35.4568 - 1.3863, 31.4131 - 1.3863]) # in log-proba 

np.float64(35.59797400581464)

Notice that this number is the same as $V_f(g)$.

In [20]:
import numpy as np

def normalize_log_probs(log_probs_vector):
    """
    Normalizes a vector of log probabilities.

    Args:
        log_probs_vector (array-like): An array-like object (e.g., list, numpy array) 
                                       of unnormalized log probabilities.

    Returns:
        numpy.ndarray: A NumPy array containing the normalized log probabilities.
                       The sum of exp(normalized_log_probs) will be approximately 1.0 
                       (within floating-point precision).
                       If the sum of exp(input log_probs_vector) is 0 
                       (e.g., all inputs are -np.inf or the input is empty),
                       this function returns an array of -np.inf of the same shape as input.
    """
    # Ensure input is a numpy array and use float64 for precision.
    # This is consistent with numerical precision often used in log-space computations.
    log_probs_vector = np.array(log_probs_vector, dtype=np.float64)

    # Calculate the log of the sum of probabilities.
    # The log_sum_exp function is assumed to be defined in the notebook's 
    # global scope (e.g., from execution of a previous cell).
    log_normalization_constant = log_sum_exp(log_probs_vector)

    if log_normalization_constant == -np.inf:
        # This case occurs if all input log_probs were -np.inf,
        # or if the input vector was empty (log_sum_exp returns -np.inf for empty input).
        # In such scenarios, the "normalized" probabilities are effectively 0,
        # so their log probabilities are -np.inf.
        return np.full_like(log_probs_vector, -np.inf, dtype=np.float64)
    else:
        # Subtract the log normalization constant from each log probability.
        # This is equivalent to dividing each probability P_i by the sum of all probabilities Sum(P_j),
        # in log space: log(P_i / Sum(P_j)) = log(P_i) - log(Sum(P_j)).
        normalized_log_probs = log_probs_vector - log_normalization_constant
        return normalized_log_probs


---
Do not care about the follows:

In [21]:
original_states_probs = normalize_log_probs([35.8177, 36.2243, 35.4568, 31.4131]) # log-probs = -values
original_states_kv = [39, 40, 41, 42]
print("log-probs (NOT values): ", original_states_probs)
print("k_v@origin: ", original_states_kv)

log-probs (NOT values):  [-1.16657401 -0.75997401 -1.52747401 -5.57117401]
k_v@origin:  [39, 40, 41, 42]


In [22]:
np.sum(np.exp([-1.1666, -0.76, -1.5275, -5.5712]))

np.float64(0.9999740061524838)

---
# Sampling statistics

In [3]:
def analyze_trajectory_occurrences(trajectory_file_path: str, return_type='probability'):
    """
    Analyze the occurrence (count or probability) of each route in the generated trajectories.
    
    Args:
        trajectory_file_path (str): Path to the trajectory file containing one trajectory per line
        return_type (str): Either 'count' for raw counts or 'probability' for normalized probabilities
        
    Returns:
        dict: Dictionary mapping trajectory strings to their occurrence counts or probabilities
    """
    import os
    from collections import Counter
    
    if not os.path.exists(trajectory_file_path):
        print(f"Trajectory file not found: {trajectory_file_path}")
        return {}
    
    # Read all trajectories from the file
    trajectories = []
    with open(trajectory_file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line:  # Skip empty lines
                trajectories.append(line)
    
    if not trajectories:
        print("No trajectories found in the file.")
        return {}
    
    # Count occurrences of each unique trajectory
    trajectory_counts = Counter(trajectories)
    
    if return_type == 'count':
        return dict(trajectory_counts)
    elif return_type == 'probability':
        total_trajectories = len(trajectories)
        trajectory_probabilities = {
            trajectory: count / total_trajectories 
            for trajectory, count in trajectory_counts.items()
        }
        return trajectory_probabilities
    else:
        raise ValueError("return_type must be either 'count' or 'probability'")

def print_trajectory_analysis(trajectory_file_path: str, top_n: int = 10, return_type='probability'):
    """
    Print a formatted analysis of trajectory occurrences.
    
    Args:
        trajectory_file_path (str): Path to the trajectory file
        top_n (int): Number of top trajectories to display
        return_type (str): Either 'count' for raw counts or 'probability' for normalized probabilities
    """
    results = analyze_trajectory_occurrences(trajectory_file_path, return_type)
    
    if not results:
        return
    
    # Sort by occurrence (descending)
    sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
    
    total_unique = len(sorted_results)
    total_trajectories = sum(results.values()) if return_type == 'count' else len(open(trajectory_file_path).readlines())
    
    print(f"Trajectory Analysis Results:")
    print(f"Total trajectories: {total_trajectories}")
    print(f"Unique trajectories: {total_unique}")
    print(f"Diversity ratio: {total_unique/total_trajectories:.3f}")
    print()
    
    unit = "Probability" if return_type == 'probability' else "Count"
    print(f"Top {min(top_n, len(sorted_results))} most frequent trajectories:")
    print(f"{'Rank':<5} {unit:<12} {'Trajectory'}")
    print("-" * 80)
    
    for i, (trajectory, value) in enumerate(sorted_results[:top_n], 1):
        if return_type == 'probability':
            value_str = f"{value:.4f}"
        else:
            value_str = str(value)
        
        # Truncate very long trajectories for display
        display_trajectory = trajectory
        if len(trajectory) > 60:
            waypoints = trajectory.split()
            if len(waypoints) > 8:
                display_trajectory = " ".join(waypoints[:4]) + " ... " + " ".join(waypoints[-4:])
        
        print(f"{i:<5} {value_str:<12} {display_trajectory}")

def analyze_waypoint_frequencies(trajectory_file_path: str, exclude_endpoints=True):
    """
    Analyze the frequency of individual waypoints across all trajectories.
    
    Args:
        trajectory_file_path (str): Path to the trajectory file
        exclude_endpoints (bool): Whether to exclude LEMD and EGLL from the analysis
        
    Returns:
        dict: Dictionary mapping waypoint names to their occurrence counts
    """
    import os
    from collections import Counter
    
    if not os.path.exists(trajectory_file_path):
        print(f"Trajectory file not found: {trajectory_file_path}")
        return {}
    
    waypoint_counts = Counter()
    total_trajectories = 0
    
    with open(trajectory_file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                total_trajectories += 1
                waypoints = line.split()
                
                # Optionally exclude endpoints
                if exclude_endpoints and len(waypoints) >= 2:
                    waypoints = waypoints[1:-1]  # Remove first and last waypoints
                
                waypoint_counts.update(waypoints)
    
    return dict(waypoint_counts), total_trajectories

def print_waypoint_analysis(trajectory_file_path: str, top_n: int = 15, exclude_endpoints=True):
    """
    Print a formatted analysis of waypoint frequencies.
    
    Args:
        trajectory_file_path (str): Path to the trajectory file
        top_n (int): Number of top waypoints to display
        exclude_endpoints (bool): Whether to exclude LEMD and EGLL from the analysis
    """
    waypoint_counts, total_trajectories = analyze_waypoint_frequencies(trajectory_file_path, exclude_endpoints)
    
    if not waypoint_counts:
        return
    
    # Sort by frequency (descending)
    sorted_waypoints = sorted(waypoint_counts.items(), key=lambda x: x[1], reverse=True)
    
    print(f"Waypoint Frequency Analysis:")
    print(f"Total trajectories analyzed: {total_trajectories}")
    print(f"Endpoints {'excluded' if exclude_endpoints else 'included'}")
    print()
    
    print(f"Top {min(top_n, len(sorted_waypoints))} most frequent waypoints:")
    print(f"{'Rank':<5} {'Waypoint':<15} {'Count':<8} {'Frequency':<10}")
    print("-" * 45)
    
    for i, (waypoint, count) in enumerate(sorted_waypoints[:top_n], 1):
        frequency = count / total_trajectories
        print(f"{i:<5} {waypoint:<15} {count:<8} {frequency:.3f}")

# Example usage
trajectory_file = '../data/graph/trajectories/LEMD_EGLL_2023_04_01_CLB_trajectories.txt'

print("=== TRAJECTORY OCCURRENCE ANALYSIS ===")
print_trajectory_analysis(trajectory_file, top_n=10, return_type='probability')

print("\n" + "="*60 + "\n")

print("=== WAYPOINT FREQUENCY ANALYSIS ===")
print_waypoint_analysis(trajectory_file, top_n=15, exclude_endpoints=True)


=== TRAJECTORY OCCURRENCE ANALYSIS ===
Trajectory Analysis Results:
Total trajectories: 1000
Unique trajectories: 217
Diversity ratio: 0.217

Top 10 most frequent trajectories:
Rank  Probability  Trajectory
--------------------------------------------------------------------------------
1     0.2470       LEMD OSTIX MOKOR KOTEM EGTD MODMI EGLL
2     0.1280       LEMD LERM OSTIX MOKOR KOTEM EGTD MODMI EGLL
3     0.0390       LEMD RBO SUSOS_49 NUBLO BAKUP AZFIC KETIK MID MAXIT EGLL
4     0.0320       LEMD BASIM CALCE DOSUL_89 ... KETIK MID MAXIT EGLL
5     0.0240       LEMD OSTIX_95 BELEN BALDA ... DESNA EGTD MODMI EGLL
6     0.0220       LEMD RBO SUSOS_49 NOVAN ... DESNA EGTD MODMI EGLL
7     0.0150       LEMD EDIGO_39 ETPAR PILIP ... OKSAW_11 ALHAD BIG_27 EGLL
8     0.0140       LEMD BASIM CALCE XORNA_80 ... KETIK MID MAXIT EGLL
9     0.0140       LEMD RBO SUSOS_49 RAVIG DESNA EGTD MODMI EGLL
10    0.0130       LEMD BASIM CALCE BLV DIKRO ELDOP ETVAX PAWDE EGKN EGLL


=== WAYPOINT FREQU

In [4]:
def analyze_route_segments(trajectory_file_path: str, segment_length: int = 2):
    """
    Analyze the frequency of route segments (consecutive waypoint sequences) in trajectories.
    
    Args:
        trajectory_file_path (str): Path to the trajectory file
        segment_length (int): Length of segments to analyze (e.g., 2 for pairs, 3 for triplets)
        
    Returns:
        dict: Dictionary mapping segment tuples to their occurrence counts
    """
    import os
    from collections import Counter
    
    if not os.path.exists(trajectory_file_path):
        print(f"Trajectory file not found: {trajectory_file_path}")
        return {}
    
    segment_counts = Counter()
    
    with open(trajectory_file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                waypoints = line.split()
                # Extract all segments of the specified length
                for i in range(len(waypoints) - segment_length + 1):
                    segment = tuple(waypoints[i:i + segment_length])
                    segment_counts[segment] += 1
    
    return dict(segment_counts)

def print_segment_analysis(trajectory_file_path: str, segment_length: int = 2, top_n: int = 15):
    """
    Print a formatted analysis of route segment frequencies.
    
    Args:
        trajectory_file_path (str): Path to the trajectory file
        segment_length (int): Length of segments to analyze
        top_n (int): Number of top segments to display
    """
    segment_counts = analyze_route_segments(trajectory_file_path, segment_length)
    
    if not segment_counts:
        return
    
    # Sort by frequency (descending)
    sorted_segments = sorted(segment_counts.items(), key=lambda x: x[1], reverse=True)
    
    total_segments = sum(segment_counts.values())
    
    print(f"Route Segment Analysis (length {segment_length}):")
    print(f"Total segments: {total_segments}")
    print(f"Unique segments: {len(sorted_segments)}")
    print()
    
    print(f"Top {min(top_n, len(sorted_segments))} most frequent segments:")
    print(f"{'Rank':<5} {'Count':<8} {'Frequency':<10} {'Segment'}")
    print("-" * 60)
    
    for i, (segment, count) in enumerate(sorted_segments[:top_n], 1):
        frequency = count / total_segments
        segment_str = " -> ".join(segment)
        print(f"{i:<5} {count:<8} {frequency:.3f}     {segment_str}")

def get_trajectory_statistics(trajectory_file_path: str):
    """
    Get basic statistics about the trajectories.
    
    Args:
        trajectory_file_path (str): Path to the trajectory file
        
    Returns:
        dict: Dictionary containing various statistics
    """
    import os
    import numpy as np
    
    if not os.path.exists(trajectory_file_path):
        print(f"Trajectory file not found: {trajectory_file_path}")
        return {}
    
    trajectory_lengths = []
    unique_trajectories = set()
    
    with open(trajectory_file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                waypoints = line.split()
                trajectory_lengths.append(len(waypoints))
                unique_trajectories.add(line)
    
    if not trajectory_lengths:
        return {}
    
    stats = {
        'total_trajectories': len(trajectory_lengths),
        'unique_trajectories': len(unique_trajectories),
        'diversity_ratio': len(unique_trajectories) / len(trajectory_lengths),
        'avg_length': np.mean(trajectory_lengths),
        'median_length': np.median(trajectory_lengths),
        'min_length': np.min(trajectory_lengths),
        'max_length': np.max(trajectory_lengths),
        'std_length': np.std(trajectory_lengths)
    }
    
    return stats

def print_trajectory_statistics(trajectory_file_path: str):
    """
    Print comprehensive statistics about the trajectories.
    """
    stats = get_trajectory_statistics(trajectory_file_path)
    
    if not stats:
        return
    
    print("Trajectory Statistics:")
    print("=" * 30)
    print(f"Total trajectories:     {stats['total_trajectories']}")
    print(f"Unique trajectories:    {stats['unique_trajectories']}")
    print(f"Diversity ratio:        {stats['diversity_ratio']:.3f}")
    print()
    print("Trajectory Length Statistics:")
    print(f"Average length:         {stats['avg_length']:.2f}")
    print(f"Median length:          {stats['median_length']:.1f}")
    print(f"Min length:             {stats['min_length']}")
    print(f"Max length:             {stats['max_length']}")
    print(f"Standard deviation:     {stats['std_length']:.2f}")

# Run additional analyses
print("=== TRAJECTORY STATISTICS ===")
print_trajectory_statistics(trajectory_file)

print("\n" + "="*60 + "\n")

print("=== ROUTE SEGMENT ANALYSIS (Pairs) ===")
print_segment_analysis(trajectory_file, segment_length=2, top_n=10)

print("\n" + "="*60 + "\n")

print("=== ROUTE SEGMENT ANALYSIS (Triplets) ===")
print_segment_analysis(trajectory_file, segment_length=3, top_n=10)


=== TRAJECTORY STATISTICS ===
Trajectory Statistics:
Total trajectories:     1000
Unique trajectories:    217
Diversity ratio:        0.217

Trajectory Length Statistics:
Average length:         10.94
Median length:          9.0
Min length:             7
Max length:             25
Standard deviation:     4.54


=== ROUTE SEGMENT ANALYSIS (Pairs) ===
Route Segment Analysis (length 2):
Total segments: 9935
Unique segments: 379

Top 10 most frequent segments:
Rank  Count    Frequency  Segment
------------------------------------------------------------
1     479      0.048     EGTD -> MODMI
2     438      0.044     MODMI -> EGLL
3     375      0.038     OSTIX -> MOKOR
4     375      0.038     MOKOR -> KOTEM
5     375      0.038     KOTEM -> EGTD
6     260      0.026     DET -> EGLL
7     247      0.025     LEMD -> OSTIX
8     245      0.025     LEMD -> EDIGO_39
9     245      0.025     EDIGO_39 -> ETPAR
10    245      0.025     ETPAR -> PILIP


=== ROUTE SEGMENT ANALYSIS (Triplets) ===
Ro